In [22]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

# Init

In [23]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [24]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [25]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [26]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [27]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-4000])
henon_test_scaled = henon_scaler.transform(henon_dataset[-4000:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [28]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [29]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [30]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = (total_plots + cols - 1) // cols
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

In [31]:
def create_stiffness_matrix(node_positions, connections):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for conn in connections:
        node_conn = conn[:2].astype(int)
        node_pos = node_positions[node_conn]
        k_val = conn[2]

        diff_vec = np.diff(node_pos, axis=0).flatten()
        unit_dir = diff_vec / np.linalg.norm(diff_vec)

        sub_block = np.outer(unit_dir, unit_dir)
        k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])

        global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")

        for local_row, global_row in enumerate(global_indices):
            for local_col, global_col in enumerate(global_indices):
                K[global_row, global_col] += k_local[local_row, local_col]

    return K

In [32]:
@njit
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [33]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
):
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors="magenta"
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    frame_tracker = 0
    test = True

    def update_springs(canvas):
        nonlocal frame_tracker, test
        # if not test:
        #     return
        # test = False
        frame_tracker = (frame_tracker + frames_moved) % steps

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]
        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

# Grid

In [34]:
N = 20

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [35]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = rng.integers(low=0, high=num_nodes, size=(henon_scaled.shape[1]))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [36]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_slices = [
    np.s_[:, :-1],
    np.s_[:-1, :],
    np.s_[:-1:2, :-1],
    np.s_[1:-1:2, 1:]
]

dst_slices = [
    np.s_[:, 1:],
    np.s_[1:, :],
    np.s_[1::2, 1:],
    np.s_[2::2, :-1]
]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [37]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir, .1, matrix_size, M_INV, DAMP, K, U
)

displacement = displacement[transient_steps_reservoir:]
velocity = velocity[transient_steps_reservoir:]

X = np.column_stack((displacement, velocity))
Y = henon_scaled[transient_steps_reservoir:]

In [39]:
henon_scaled[transient_steps_reservoir + tau_steps :]

array([[ 0.48849597,  0.06650708],
       [ 0.43183561,  0.48856167],
       [ 0.65210564,  0.43189946],
       ...,
       [ 0.47949949, -1.36867371],
       [ 0.0165981 ,  0.4795649 ],
       [ 1.13845524,  0.01664843]], shape=(20000, 2))

In [42]:
X_data

array([[-0.0099922 ,  0.02917136, -0.00990147, ..., -0.00104014,
        -0.00851336,  0.00258104],
       [-0.00983139,  0.02834805, -0.00974895, ...,  0.00085753,
        -0.00839167,  0.00208919],
       [-0.00963876,  0.02755978, -0.00954917, ...,  0.00196346,
        -0.00873966,  0.00156065],
       ...,
       [ 0.02516718, -0.1494784 ,  0.02504822, ...,  0.01046623,
        -0.00471727, -0.0028601 ],
       [ 0.02519518, -0.14933109,  0.02512112, ...,  0.01178666,
        -0.00382942, -0.0027494 ],
       [ 0.02524868, -0.14915364,  0.02521637, ...,  0.0111143 ,
        -0.00257428, -0.00250227]], shape=(17999, 1600))

In [41]:
X_data = X[transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

x_scaler = StandardScaler()
X_train, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

model = RidgeCV()
model.fit(X, Y)

# Y_pred_scaled = model.predict(disp[])
# Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
# Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

ValueError: Found input variables with inconsistent numbers of samples: [20000, 20001]